# 02 - Density Comparison (Multiclass)

Notebook para comparar **densidades reais vs MoSS** em cenários multiclasse.

- Datasets: `/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/multiclass`
- MoSS: `/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/moss/multiclass`
- Busca automática por `moss_d*` por número de classes (`.pkl` e fallback `.csv`).

A lógica segue o binário, mas por classe: batch aleatório UPP, KDE por classe e melhor match por média de KDE-L1.

In [ ]:
import os
import ast
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mlquantify.model_selection import UPP

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
np.random.seed(SEED)

DATASETS_ROOT = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/multiclass"
MOSS_ROOT = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/moss/multiclass"

BATCH_SIZE = 100
GRID_SIZE = 200
TOPK = 5
MAX_CURVES = None  # None = usa todas

grid = np.linspace(0, 1, GRID_SIZE)

assert os.path.isdir(DATASETS_ROOT), f"Diretório não encontrado: {DATASETS_ROOT}"
assert os.path.isdir(MOSS_ROOT), f"Diretório não encontrado: {MOSS_ROOT}"

In [ ]:
def kde_curve(scores, grid):
    s = np.asarray(scores, dtype=float)
    if len(s) < 2 or np.allclose(np.std(s), 0.0):
        return np.zeros_like(grid)
    try:
        return gaussian_kde(s)(grid)
    except Exception:
        return np.zeros_like(grid)


def mean_l1_multiclass(real_kdes, moss_kdes):
    return float(np.mean([np.mean(np.abs(r - m)) for r, m in zip(real_kdes, moss_kdes)]))


def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    classes = np.unique(y)
    m = {c: i for i, c in enumerate(classes)}
    y = np.array([m[v] for v in y], dtype=int)
    return X, y


def random_upp_batch_idx(Xte, yte, seed=SEED):
    protocol = UPP(batch_size=min(BATCH_SIZE, len(yte)), n_prevalences=10, repeats=3, random_state=seed)
    batches = list(protocol.split(Xte, yte))
    if not batches:
        raise RuntimeError("UPP não gerou batches")
    return batches[np.random.randint(0, len(batches))]

In [ ]:
def moss_candidates(n_classes):
    return [
        os.path.join(MOSS_ROOT, f"moss_d_lite_{n_classes}.pkl"),
        os.path.join(MOSS_ROOT, f"moss_d_{n_classes}.pkl"),
        os.path.join(MOSS_ROOT, f"moss_d_lite_{n_classes}.csv"),
        os.path.join(MOSS_ROOT, f"moss_d_{n_classes}.csv"),
    ]


def iter_curves_from_pkl(path):
    with open(path, 'rb') as f:
        moss = pickle.load(f)
    for (alpha_prev, merge), curves in moss.items():
        for curve_id, scores in enumerate(curves):
            yield dict(alpha_prev=tuple(float(x) for x in alpha_prev), merge=float(merge), curve_id=int(curve_id), scores=np.asarray(scores))


def iter_curves_from_csv(path, n_classes):
    df = pd.read_csv(path)
    if not {'prev','merge','curve_id'}.issubset(df.columns):
        return
    score_cols = [f'scores_{c}' for c in range(n_classes)]
    if not all(c in df.columns for c in score_cols):
        return

    for _, row in df.iterrows():
        try:
            alpha_prev = tuple(float(x) for x in ast.literal_eval(str(row['prev'])))
            merge = float(row['merge'])
            curve_id = int(row['curve_id'])
            per_class = [np.asarray(ast.literal_eval(str(row[c])), dtype=float) for c in score_cols]
            n = min(len(a) for a in per_class)
            if n < 2:
                continue
            scores = np.column_stack([a[:n] for a in per_class])
            yield dict(alpha_prev=alpha_prev, merge=merge, curve_id=curve_id, scores=scores)
        except Exception:
            continue


def load_curve_iterator(n_classes):
    for path in moss_candidates(n_classes):
        if not os.path.exists(path):
            continue
        if path.endswith('.pkl'):
            return iter_curves_from_pkl(path), path
        return iter_curves_from_csv(path, n_classes), path
    return None, None


def best_moss_match(real_kdes, n_classes):
    it, used = load_curve_iterator(n_classes)
    if it is None:
        return None

    best = dict(best_l1=np.inf, best_prev=None, best_merge=None, best_curve_id=None, best_kdes=None, moss_file=used)
    seen = 0
    for item in it:
        scores = np.asarray(item['scores'])
        if scores.ndim != 2 or scores.shape[1] != n_classes:
            continue
        if MAX_CURVES is not None and seen >= MAX_CURVES:
            break
        seen += 1

        moss_kdes = [kde_curve(scores[:, c], grid) for c in range(n_classes)]
        l1 = mean_l1_multiclass(real_kdes, moss_kdes)
        if l1 < best['best_l1']:
            best.update(best_l1=float(l1), best_prev=item['alpha_prev'], best_merge=float(item['merge']), best_curve_id=int(item['curve_id']), best_kdes=moss_kdes)

    return best if np.isfinite(best['best_l1']) else None

In [ ]:
rows, plot_cache = [], []
datasets = sorted([f for f in os.listdir(DATASETS_ROOT) if f.endswith('.csv')])
if not datasets:
    raise RuntimeError(f"Nenhum CSV encontrado em {DATASETS_ROOT}")

for ds in datasets:
    X, y = load_dataset(os.path.join(DATASETS_ROOT, ds))
    n_classes = len(np.unique(y))

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, stratify=y, random_state=SEED)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr)

    idx = random_upp_batch_idx(Xte, yte)
    scores_batch = clf.predict_proba(Xte[idx])
    real_kdes = [kde_curve(scores_batch[:, c], grid) for c in range(n_classes)]

    best = best_moss_match(real_kdes, n_classes)
    if best is None:
        rows.append(dict(dataset=ds, n_classes=n_classes, best_moss_prev=None, best_moss_merge=None, best_curve_id=None, best_kde_l1=np.nan, moss_file=None, status='moss_not_found_or_invalid'))
        continue

    rows.append(dict(dataset=ds, n_classes=n_classes, best_moss_prev=best['best_prev'], best_moss_merge=best['best_merge'], best_curve_id=best['best_curve_id'], best_kde_l1=best['best_l1'], moss_file=best['moss_file'], status='ok'))
    plot_cache.append(dict(dataset=ds, n_classes=n_classes, real_kdes=real_kdes, moss_kdes=best['best_kdes'], best_l1=best['best_l1'], best_prev=best['best_prev'], best_merge=best['best_merge'], best_curve_id=best['best_curve_id']))

summary_best = pd.DataFrame(rows).sort_values('best_kde_l1', na_position='last')
summary_best.head(20)

In [ ]:
out_dir = os.path.join(os.getcwd(), 'results', 'exp_015_density_multiclass')
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, 'density_multiclass_summary.csv')
summary_best.to_csv(out_csv, index=False)
print('Resumo salvo em:', out_csv)
print('Datasets avaliados:', len(summary_best))
print('Datasets com match válido:', int((summary_best['status'] == 'ok').sum()))

In [ ]:
valid = [r for r in plot_cache if np.isfinite(r['best_l1'])]
if not valid:
    raise RuntimeError('Nenhum dataset com match válido para plotar')

valid_sorted = sorted(valid, key=lambda r: r['best_l1'])
best_group = valid_sorted[:TOPK]
worst_group = valid_sorted[-TOPK:]


def plot_group(group, title):
    for row in group:
        nc = row['n_classes']
        fig, axes = plt.subplots(nc, 1, figsize=(9, 2.7*nc), sharex=True)
        if nc == 1:
            axes = [axes]
        for c, ax in enumerate(axes):
            ax.plot(grid, row['real_kdes'][c], label=f'Real c={c}', linewidth=2)
            ax.plot(grid, row['moss_kdes'][c], label=f'MoSS c={c}', linewidth=2)
            ax.set_ylabel('Densidade')
            ax.legend(loc='upper right')
        axes[-1].set_xlabel('Score')
        fig.suptitle(f"{title} | {row['dataset']} | classes={nc} | L1={row['best_l1']:.4f} | prev={row['best_prev']} | merge={row['best_merge']:.4f} | curve={row['best_curve_id']}", y=1.01, fontsize=10)
        plt.tight_layout()
        plt.show()

plot_group(best_group, 'Mais próximos do MoSS')
plot_group(worst_group, 'Mais distantes do MoSS')

In [ ]:
summary_ok = summary_best[summary_best['status'] == 'ok'].copy()
if len(summary_ok) > 0:
    display(
        summary_ok.groupby('n_classes', as_index=False)
        .agg(n_datasets=('dataset', 'count'), mean_kde_l1=('best_kde_l1', 'mean'), median_kde_l1=('best_kde_l1', 'median'))
        .sort_values('mean_kde_l1')
    )

summary_best.head(30)